# CrisisLens · Custom CNN Training (Kaggle GPU)

6 類災情分類，資料集：MEDIC (QCRI)。訓練自建 CNN（Earthquake/Flood/Fire/Typhoon/Landslide/Other），並跑 3 個 ablation 對照（no-BN / big-FC / shallow）。

## 執行前
1. Notebook options → Accelerator → GPU T4 x2（或 P100）
2. Internet → On（需下載 HuggingFace QCRI/MEDIC 資料集）
3. 確認下方 Config cell 的 `HF_DATASET`、`MAX_SAMPLES_PER_CLASS`

## 輸出
- `custom_cnn.pth` — v1 final 權重（~1.5 MB）
- `custom_cnn_classes.json` — 6 類對照（含 MEDIC_MAP、test_acc）
- `training_curves.png`、`confusion_matrix.png`

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "datasets"], check=True)

import torch
print(f"Python:     {sys.version.split()[0]}")
print(f"PyTorch:    {torch.__version__}")
print(f"CUDA:       {torch.version.cuda}")
print(f"GPU 可用:   {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU 名稱:   {torch.cuda.get_device_name(0)}")
    print(f"GPU 記憶體: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print("\n⚠️ 沒偵測到 GPU — 請從右側 Notebook options 啟用 GPU 後 Restart Session")

## 1. Config

In [ ]:
from pathlib import Path

# ── 資料來源：HuggingFace QCRI/MEDIC（Kaggle 需開 internet）────
HF_DATASET = "QCRI/MEDIC"

# ── 輸出路徑（Kaggle working dir，跑完從右側 Output 下載）─────
OUT_DIR      = Path("/kaggle/working")
WEIGHTS_PATH = OUT_DIR / "custom_cnn.pth"
MAPPING_PATH = OUT_DIR / "custom_cnn_classes.json"
CURVES_PATH  = OUT_DIR / "training_curves.png"
CM_PATH      = OUT_DIR / "confusion_matrix.png"

# ── 訓練超參數 ─────────────────────────────────────────────
BATCH_SIZE    = 64
LEARNING_RATE = 1e-3
EPOCHS        = 30
NUM_WORKERS   = 4          # Kaggle 約 4 核，吃滿加速資料載入
SEED          = 42
USE_DATA_PARALLEL = False  # True 則用 nn.DataParallel 吃滿 T4×2（可選加速）
MAX_SAMPLES_PER_CLASS = None  # train 每類抽樣上限（控訓練時間）；None = 全量
MAX_EVAL_PER_CLASS    = None  # val/test 每類抽樣上限（試跑可設小如 200 加速）；None = 全量
CACHE_IN_MEMORY       = True   # 影像解碼一次存記憶體（提高 GPU 使用率）；全量資料若 RAM 不足設 False

# ── 6 類正準順序（對齊 utils/config.py，index 不可改）──────────
CLASSES_EN = [
    "Earthquake Damage", "Flood", "Fire",
    "Typhoon or Storm Damage", "Landslide", "Other or No Disaster",
]
CLASSES_ZH = [
    "地震或建築損壞", "淹水", "火災",
    "颱風或強風災損", "土石流或坍方", "其他或無明顯災害",
]
NUM_CLASSES = len(CLASSES_EN)  # 6

# ── MEDIC disaster_types 原始標籤 → 6 類 index（7→6，合併 not/other）──
MEDIC_MAP = {
    "earthquake":     0,
    "flood":          1,
    "fire":           2,
    "hurricane":      3,
    "landslide":      4,
    "not_disaster":   5,
    "other_disaster": 5,
}

print(f"HF_DATASET: {HF_DATASET}")
print(f"NUM_CLASSES: {NUM_CLASSES}  -> {CLASSES_EN}")


## 2. Imports

In [ ]:
import os, json, time, copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, WeightedRandomSampler, Dataset
from torchvision import transforms
from datasets import load_dataset
from PIL import ImageFile
ImageFile.LOAD_TRUNCATED_IMAGES = True
from sklearn.metrics import classification_report, confusion_matrix
import matplotlib.pyplot as plt
import seaborn as sns

torch.manual_seed(SEED)
np.random.seed(SEED)

## 3. 載入 MEDIC 並檢視類別分布

In [ ]:
# （可選）HF 認證：在 Kaggle 設好名為 HF_TOKEN 的 Secret 後會自動帶入，
# 避免未認證的速率限制、加快 MEDIC 下載。沒設則照常以未認證方式下載。
try:
    from kaggle_secrets import UserSecretsClient
    _hf_tok = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = _hf_tok          # datasets / huggingface_hub 會自動讀取
    from huggingface_hub import login
    login(token=_hf_tok, add_to_git_credential=False)
    print("✅ 已以 HF_TOKEN 認證")
except Exception as e:
    print(f"⚠️ 未使用 HF_TOKEN（{type(e).__name__}），改以未認證下載（較慢、有速率限制）")

# 載入 MEDIC（非串流，完整下載+快取，支援多 epoch 隨機抽樣）
raw = load_dataset(HF_DATASET)
print(raw)

# 取得 disaster_types 的整數 label → 原始字串名
SPLIT_TRAIN = "train"
SPLIT_DEV   = "validation" if "validation" in raw else ("dev" if "dev" in raw else None)
SPLIT_TEST  = "test" if "test" in raw else None
assert SPLIT_DEV is not None and SPLIT_TEST is not None, f"找不到 dev/test split：{list(raw.keys())}"

DT_NAMES = raw[SPLIT_TRAIN].features["disaster_types"].names
print("MEDIC disaster_types 原始類別：", DT_NAMES)

# 確認 MEDIC_MAP 覆蓋所有原始標籤（避免漏映射）
missing = [n for n in DT_NAMES if n not in MEDIC_MAP]
assert not missing, f"MEDIC_MAP 未覆蓋：{missing}"

# 檢視一筆樣本結構
ex = raw[SPLIT_TRAIN][0]
print("樣本欄位：", list(ex.keys()))
print("image 型別：", type(ex["image"]))

## 4. Transforms (Train 用 augmentation，Val 用單純 resize)

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.3),
    transforms.RandomRotation(15),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])
val_tf = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
])

## 5. Dataset & DataLoader

In [ ]:
class MedicDataset(Dataset):
    """把 MEDIC HF split 包成 (image_tensor, label6)；套 MEDIC_MAP（7→6）。

    每個 split 各自持有獨立 transform —— 不共享底層 dataset，
    因此不會有舊版「val transform 覆蓋 train augmentation」的副作用。
    cache=True 時於 __init__ 將影像解碼並縮到 <=256 存記憶體，
    之後每 epoch 只做 augmentation，免重複 JPEG 解碼（提高 GPU 使用率）。
    """
    def __init__(self, hf_split, transform, max_per_class=None, cache=False):
        self.hf_split  = hf_split
        self.transform = transform
        self.samples   = []                 # list of (hf_index, label6)
        self.counts    = [0] * NUM_CLASSES
        names = hf_split.features["disaster_types"].names
        for i, dt in enumerate(hf_split["disaster_types"]):
            label6 = MEDIC_MAP.get(names[dt])
            if label6 is None:
                continue
            # 注意：max_per_class 取每類在原始 dataset 的「前 N 筆」（未洗牌），純為控訓練時間；若在意分布偏差可改先洗牌再截取。
            if max_per_class is not None and self.counts[label6] >= max_per_class:
                continue
            self.counts[label6] += 1
            self.samples.append((i, label6))

        # 可選：解碼一次存記憶體（搭配 num_workers=0，避免多進程複製快取）
        self.images = None
        if cache:
            self.images = []
            for hf_i, _ in self.samples:
                im = self.hf_split[hf_i]["image"].convert("RGB")
                im.thumbnail((256, 256))   # 等比縮到 <=256 邊長，省記憶體
                self.images.append(im.copy())
            mb = sum(im.size[0] * im.size[1] * 3 for im in self.images) / 1e6
            print(f"  已快取 {len(self.images)} 張影像於記憶體（約 {mb:.0f} MB）")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        if self.images is not None:
            img = self.images[idx]
        else:
            hf_i, _ = self.samples[idx]
            img = self.hf_split[hf_i]["image"].convert("RGB")
        return self.transform(img), self.samples[idx][1]


train_ds = MedicDataset(raw[SPLIT_TRAIN], train_tf, max_per_class=MAX_SAMPLES_PER_CLASS, cache=CACHE_IN_MEMORY)
val_ds   = MedicDataset(raw[SPLIT_DEV],   val_tf,  max_per_class=MAX_EVAL_PER_CLASS, cache=CACHE_IN_MEMORY)
test_ds  = MedicDataset(raw[SPLIT_TEST],  val_tf,  max_per_class=MAX_EVAL_PER_CLASS, cache=CACHE_IN_MEMORY)

print("Train 類別分布：")
for i, (en, n) in enumerate(zip(CLASSES_EN, train_ds.counts)):
    print(f"  [{i}] {en:25s} {n:6d}")
print(f"Train: {len(train_ds)}  Dev: {len(val_ds)}  Test: {len(test_ds)}")

# sqrt-inverse 類別權重 → WeightedRandomSampler（處理嚴重不平衡）
counts   = np.array(train_ds.counts, dtype=float)
class_w  = 1.0 / np.sqrt(np.maximum(counts, 1.0))
sample_w = [class_w[label] for _, label in train_ds.samples]
sampler  = WeightedRandomSampler(sample_w, num_samples=len(sample_w), replacement=True)

# 快取在記憶體時用單進程（避免每個 worker 複製整份快取）；否則開多 worker 平行解碼
if CACHE_IN_MEMORY:
    _loader_kw = dict(num_workers=0, pin_memory=True)
else:
    _loader_kw = dict(num_workers=NUM_WORKERS, pin_memory=True)
    if NUM_WORKERS > 0:
        _loader_kw.update(persistent_workers=True, prefetch_factor=4)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, sampler=sampler, **_loader_kw)
val_loader   = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)
test_loader  = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False, **_loader_kw)


## 6. v1 (Baseline) — 4-block CNN with BatchNorm + GAP

**⚠️ 注意**：這個 class 定義必須與本地 `models/custom_cnn_model.py` 的 `DisasterCNN_v1` 字字相同。
若架構不一致，訓練好的 `.pth` 無法在本地 streamlit 載入（layer name mismatch）。

In [ ]:
class DisasterCNN_v1(nn.Module):
    """4-block CNN baseline for 6-class disaster classification."""
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            # Block 1: 224 -> 112
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 2: 112 -> 56
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 3: 56 -> 28
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            # Block 4: 28 -> 14
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1),
            nn.Flatten(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

### 訓練 v1（記錄訓練曲線，保存 best by Val Acc）

In [ ]:
def train_one_model(model_class, model_name: str, epochs: int = EPOCHS):
    """通用訓練函式 — 回傳 (history, best_val_acc, best_state)。best_state 已去除 DataParallel 包裝。"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model     = model_class(NUM_CLASSES).to(device)
    if USE_DATA_PARALLEL and torch.cuda.device_count() > 1:
        model = nn.DataParallel(model)
    loss_fn   = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler    = torch.amp.GradScaler("cuda") if device == "cuda" else None

    history      = {"train_loss": [], "val_loss": [], "val_acc": []}
    best_val_acc = 0.0
    best_state   = None

    core = model.module if isinstance(model, nn.DataParallel) else model
    n_params = sum(p.numel() for p in core.parameters() if p.requires_grad)
    print(f"\n=== {model_name} ===  可訓練參數: {n_params:,}")

    for epoch in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        train_loss = 0.0
        for imgs, labels in train_loader:
            imgs   = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            optimizer.zero_grad()
            if scaler is not None:
                with torch.amp.autocast("cuda"):
                    loss = loss_fn(model(imgs), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = loss_fn(model(imgs), labels)
                loss.backward()
                optimizer.step()
            train_loss += loss.item()
        train_loss /= len(train_loader)

        model.eval()
        val_loss = correct = total_n = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs   = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                logits = model(imgs)
                val_loss += loss_fn(logits, labels).item()
                correct += (logits.argmax(1) == labels).sum().item()
                total_n += labels.size(0)
        val_loss /= len(val_loader)
        val_acc   = correct / total_n

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["val_acc"].append(val_acc)

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = copy.deepcopy(core.state_dict())

        print(f"  Epoch {epoch:2d}/{epochs}  Train Loss: {train_loss:.4f}  "
              f"Val Loss: {val_loss:.4f}  Val Acc: {val_acc:.4f}  ({time.time()-t0:.1f}s)")
        scheduler.step()

    print(f"  → Best Val Acc: {best_val_acc:.4f}")
    return history, best_val_acc, best_state


@torch.no_grad()
def evaluate(model_class, state, loader):
    """用 best_state 在指定 loader 上評估，回傳 (preds, trues)。"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model_class(NUM_CLASSES).to(device)
    model.load_state_dict(state)
    model.eval()
    preds, trues = [], []
    for imgs, labels in loader:
        imgs = imgs.to(device, non_blocking=True)
        preds.extend(model(imgs).argmax(1).cpu().tolist())
        trues.extend(labels.tolist())
    return preds, trues


# 訓練 v1
v1_history, v1_best_acc, v1_best_state = train_one_model(DisasterCNN_v1, "v1 Baseline")


## 7. Ablation v2 — No-BN（拿掉所有 BatchNorm）

預期觀察：訓練 loss 震盪、收斂變慢、Val Acc 比 v1 下降 5-8%
學到什麼：BatchNorm 對訓練穩定性的關鍵作用

In [ ]:
class DisasterCNN_v2_NoBN(nn.Module):
    """v1 - all BatchNorm layers"""
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(256, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

v2_history, v2_best_acc, v2_best_state = train_one_model(DisasterCNN_v2_NoBN, "v2 No-BN")

## 8. Ablation v3 — Big-FC（GAP 換成 Flatten + 大 FC）

預期觀察：參數量 +100x、嚴重過擬合（Train Acc 99% / Val Acc ~50%）
學到什麼：GAP 為什麼取代 Flatten + 大 FC

In [ ]:
class DisasterCNN_v3_BigFC(nn.Module):
    """v1 with GAP replaced by Flatten + Linear(256*14*14, num_classes)"""
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(128, 256, 3, padding=1), nn.BatchNorm2d(256), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(256 * 14 * 14, num_classes),  # 50176 -> 6
        )
    def forward(self, x):
        return self.classifier(self.features(x))

v3_history, v3_best_acc, v3_best_state = train_one_model(DisasterCNN_v3_BigFC, "v3 Big-FC")

## 9. Ablation v4 — Shallow（只保留 Block 1 + Block 2）

預期觀察：Val Acc 比 v1 下降 10-15%
學到什麼：深度對特徵抽取能力的影響

In [ ]:
class DisasterCNN_v4_Shallow(nn.Module):
    """v1 with only Block 1 + Block 2 (drop Block 3 + Block 4)"""
    def __init__(self, num_classes: int = 6):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(inplace=True), nn.MaxPool2d(2),
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(inplace=True), nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.AdaptiveAvgPool2d(1), nn.Flatten(), nn.Dropout(0.3), nn.Linear(64, num_classes),
        )
    def forward(self, x):
        return self.classifier(self.features(x))

v4_history, v4_best_acc, v4_best_state = train_one_model(DisasterCNN_v4_Shallow, "v4 Shallow")

## 10. 四個變體的成績對比

In [ ]:
import pandas as pd

# 在 test 上評估 v1（部署版）以取得最終成績與混淆矩陣資料
v1_test_pred, v1_test_true = evaluate(DisasterCNN_v1, v1_best_state, test_loader)
v1_test_acc = np.mean(np.array(v1_test_pred) == np.array(v1_test_true))

results = pd.DataFrame([
    {"model": "v1 Baseline", "best_dev_acc": v1_best_acc, "params": sum(p.numel() for p in DisasterCNN_v1(NUM_CLASSES).parameters())},
    {"model": "v2 No-BN",    "best_dev_acc": v2_best_acc, "params": sum(p.numel() for p in DisasterCNN_v2_NoBN(NUM_CLASSES).parameters())},
    {"model": "v3 Big-FC",   "best_dev_acc": v3_best_acc, "params": sum(p.numel() for p in DisasterCNN_v3_BigFC(NUM_CLASSES).parameters())},
    {"model": "v4 Shallow",  "best_dev_acc": v4_best_acc, "params": sum(p.numel() for p in DisasterCNN_v4_Shallow(NUM_CLASSES).parameters())},
])
print(results.to_string(index=False))
print(f"\nv1 部署版 Test Acc: {v1_test_acc:.4f}")

## 11. 訓練曲線疊圖

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors = {"v1": "#38bdf8", "v2": "#f87171", "v3": "#fbbf24", "v4": "#a78bfa"}
for name, hist, key in [
    ("v1 Baseline", v1_history, "v1"),
    ("v2 No-BN",    v2_history, "v2"),
    ("v3 Big-FC",   v3_history, "v3"),
    ("v4 Shallow",  v4_history, "v4"),
]:
    ax1.plot(hist["val_loss"], color=colors[key], linewidth=2, label=name)
    ax2.plot(hist["val_acc"],  color=colors[key], linewidth=2, label=name, marker="o", markersize=3)

ax1.set_title("Validation Loss")
ax1.set_xlabel("Epoch"); ax1.legend(); ax1.grid(alpha=0.3)
ax2.set_title("Validation Accuracy")
ax2.set_xlabel("Epoch"); ax2.set_ylim(0, 1); ax2.legend(); ax2.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(CURVES_PATH, dpi=120, bbox_inches="tight")
plt.show()
print(f"✅ 已存: {CURVES_PATH}")

## 12. v1 Confusion Matrix + Classification Report

In [ ]:
print("=== v1 (deploy) Classification Report — TEST split ===")
print(classification_report(v1_test_true, v1_test_pred, target_names=CLASSES_EN, digits=4))

cm = confusion_matrix(v1_test_true, v1_test_pred, labels=list(range(NUM_CLASSES)))
plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=CLASSES_EN, yticklabels=CLASSES_EN)
plt.xlabel("Predicted"); plt.ylabel("True")
plt.title("v1 Confusion Matrix (MEDIC test, 6-class)")
plt.xticks(rotation=45, ha="right"); plt.yticks(rotation=0)
plt.tight_layout(); plt.savefig(CM_PATH, dpi=120); plt.show()
print("⚠️ 重點檢視：地震(0)↔土石流(4) 混淆、以及『其他/無災害(5)』是否主導預測")

## 13. 儲存 v1 (final deploy 版本) 權重

僅保存 v1 的 best state — v2/v3/v4 只在報告中比較，不 deploy。
儲存內容含 6 類標籤、MEDIC_MAP、val/test acc。

In [ ]:
torch.save(v1_best_state, WEIGHTS_PATH)
print(f"✅ 權重已存: {WEIGHTS_PATH}  ({WEIGHTS_PATH.stat().st_size / 1e6:.2f} MB)")

mapping = {
    "classes":      CLASSES_EN,
    "zh_labels":    CLASSES_ZH,
    "class_to_idx": {c: i for i, c in enumerate(CLASSES_EN)},
    "num_classes":  NUM_CLASSES,
    "architecture": "DisasterCNN_v1",
    "val_acc":      float(v1_best_acc),
    "test_acc":     float(v1_test_acc),
    "dataset":      "QCRI/MEDIC (disaster_types, 7->6)",
    "medic_map":    MEDIC_MAP,
}
with open(MAPPING_PATH, "w", encoding="utf-8") as f:
    json.dump(mapping, f, ensure_ascii=False, indent=2)
print(f"✅ 類別對照已存: {MAPPING_PATH}")
print(json.dumps(mapping, ensure_ascii=False, indent=2))

## 14. 下載連結

點下方連結下載；或從右側 Output 面板選 ⋮ → Download。

放到本地專案的路徑：
- `custom_cnn.pth` → `models/custom_cnn.pth`
- `custom_cnn_classes.json` → `models/custom_cnn_classes.json`

訓練曲線、混淆矩陣是報告素材（可選下載）。

In [ ]:
from IPython.display import FileLink, display

print("可下載檔案：\n")
for path in [WEIGHTS_PATH, MAPPING_PATH, CURVES_PATH, CM_PATH]:
    if path.exists():
        size = path.stat().st_size / 1e6
        print(f"  {path.name:35s}  {size:6.2f} MB")
        display(FileLink(str(path)))

## 15. ResNet18 遷移學習 vs 自訓 CNN 比較

載入已訓練的自訓 CNN（上傳的 `custom_cnn.pth`），與在 MEDIC 上微調的 ResNet18 在**同一個 test split** 上比較預測效果。

**執行前提**：只需先跑 Config / Imports / 載入 MEDIC / Transforms / Dataset & DataLoader / DisasterCNN_v1 定義；**不需重跑 v1~v4 訓練**。

**上傳 CNN 權重**：把 `custom_cnn.pth` 用 Kaggle「Add Data → Upload」傳成 Dataset 並 Attach，再把下方 `PRETRAINED_CNN_PATH` 改成它的 `/kaggle/input/...` 路徑。

In [ ]:
import torchvision.models as tvm

# ── 比較設定 ─────────────────────────────────────────────
PRETRAINED_CNN_PATH = "/kaggle/input/custom-cnn-6class/custom_cnn.pth"  # ← 改成你上傳的路徑
RESNET_EPOCHS = 8
RESNET_LR     = 1e-4   # 全層微調用較小 lr，避免破壞預訓練特徵

def build_resnet(num_classes):
    """ImageNet 預訓練 ResNet18，換成 num_classes 類分類頭，全層可訓練。"""
    net = tvm.resnet18(weights=tvm.ResNet18_Weights.IMAGENET1K_V1)
    net.fc = nn.Linear(net.fc.in_features, num_classes)
    return net


def train_transfer(model_fn, name, epochs, lr):
    """微調訓練（AMP + cosine + label smoothing），回傳 (best_val_acc, best_state)。"""
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model  = model_fn(NUM_CLASSES).to(device)
    opt    = torch.optim.Adam(model.parameters(), lr=lr)
    sched  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    loss_fn = nn.CrossEntropyLoss(label_smoothing=0.1)
    scaler = torch.amp.GradScaler("cuda") if device == "cuda" else None
    best_acc, best_state = 0.0, None
    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"\n=== {name} ===  可訓練參數: {n_params:,}")
    for ep in range(1, epochs + 1):
        t0 = time.time()
        model.train()
        for imgs, labels in train_loader:
            imgs = imgs.to(device, non_blocking=True)
            labels = labels.to(device, non_blocking=True)
            opt.zero_grad()
            if scaler is not None:
                with torch.amp.autocast("cuda"):
                    loss = loss_fn(model(imgs), labels)
                scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            else:
                loss = loss_fn(model(imgs), labels); loss.backward(); opt.step()
        model.eval(); correct = total = 0
        with torch.no_grad():
            for imgs, labels in val_loader:
                imgs = imgs.to(device, non_blocking=True)
                labels = labels.to(device, non_blocking=True)
                correct += (model(imgs).argmax(1) == labels).sum().item()
                total += labels.size(0)
        acc = correct / total
        if acc > best_acc:
            best_acc, best_state = acc, copy.deepcopy(model.state_dict())
        sched.step()
        print(f"  Epoch {ep:2d}/{epochs}  Val Acc: {acc:.4f}  ({time.time()-t0:.1f}s)")
    print(f"  → Best Val Acc: {best_acc:.4f}")
    return best_acc, best_state


rn_best_acc, rn_best_state = train_transfer(build_resnet, "ResNet18-FT", RESNET_EPOCHS, RESNET_LR)

In [ ]:
import os
import numpy as np
import pandas as pd

@torch.no_grad()
def eval_on_test(model):
    device = "cuda" if torch.cuda.is_available() else "cpu"
    model = model.to(device).eval()
    preds, trues = [], []
    for imgs, labels in test_loader:
        imgs = imgs.to(device, non_blocking=True)
        preds.extend(model(imgs).argmax(1).cpu().tolist())
        trues.extend(labels.tolist())
    return np.array(preds), np.array(trues)

# ResNet18（微調後）
rn = build_resnet(NUM_CLASSES); rn.load_state_dict(rn_best_state)
rn_pred, rn_true = eval_on_test(rn)
rn_acc = float((rn_pred == rn_true).mean())

# 自訓 CNN（載入上傳的權重）
if os.path.exists(PRETRAINED_CNN_PATH):
    cnn_state = torch.load(PRETRAINED_CNN_PATH, map_location="cpu", weights_only=True)
    print(f"已載入上傳的 CNN 權重：{PRETRAINED_CNN_PATH}")
else:
    cnn_state = v1_best_state   # 退路：本 run 有訓練 v1 才可用
    print("找不到上傳路徑，改用本 run 的 v1_best_state")
cnn = DisasterCNN_v1(NUM_CLASSES); cnn.load_state_dict(cnn_state)
cnn_pred, cnn_true = eval_on_test(cnn)
cnn_acc = float((cnn_pred == cnn_true).mean())

cmp = pd.DataFrame([
    {"model": "自訓 CNN (from-scratch)",  "params": sum(p.numel() for p in DisasterCNN_v1(NUM_CLASSES).parameters()), "test_acc": round(cnn_acc, 4)},
    {"model": "ResNet18 (transfer, FT)",  "params": sum(p.numel() for p in build_resnet(NUM_CLASSES).parameters()),  "test_acc": round(rn_acc, 4)},
])
print(cmp.to_string(index=False))
print("\n=== 自訓 CNN — Test ===")
print(classification_report(cnn_true, cnn_pred, target_names=CLASSES_EN, digits=4))
print("=== ResNet18-FT — Test ===")
print(classification_report(rn_true, rn_pred, target_names=CLASSES_EN, digits=4))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))
for ax, (title, yt, yp) in zip(
        axes,
        [("自訓 CNN", cnn_true, cnn_pred), ("ResNet18-FT", rn_true, rn_pred)]):
    cm = confusion_matrix(yt, yp, labels=list(range(NUM_CLASSES)))
    sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
                xticklabels=CLASSES_EN, yticklabels=CLASSES_EN, ax=ax)
    ax.set_title(f"{title} (Test Acc {float((yp==yt).mean()):.3f})")
    ax.set_xlabel("Predicted"); ax.set_ylabel("True")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()
plt.savefig("/kaggle/working/cnn_vs_resnet_cm.png", dpi=120)
plt.show()